# Plant Pathology 2020 — FGVC7

**Задача:** Классификация болезней листьев яблони (4 класса).
**Метрика:** ROC-AUC (macro-averaged).

## Пайплайн

| Stage | Что делаем | Зачем |
|---|---|---|
| **Stage 1** | 5-Fold SE-ResNeXt50, CrossEntropy, 20 эпох | Базовые модели + OOF-предсказания |
| **Stage 2** | Knowledge Distillation (мягкие метки, 10 эпох) | Сглаживание шумных меток |
| **Инференс** | 4 TTA + ансамбль 5 фолдов (веса по AUC) | Стабильные предсказания |


In [ ]:
import os, random, warnings, time, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, confusion_matrix

warnings.filterwarnings('ignore')
print("Библиотеки импортированы.")


In [ ]:
class CFG:
    seed = 42
    data_dir = '/kaggle/input/competitions/plant-pathology-2020-fgvc7'
    img_h = 320
    img_w = 512

    batch_size = 16
    epochs = 20               # Stage 1
    epochs_distill = 10       # Stage 2: дистилляция
    lr = 3e-4
    weight_decay = 1e-3

    # Knowledge Distillation
    soft_label_ratio = 0.3    # 0.3 OOF + 0.7 original

    n_folds = 5
    model_name = 'seresnext50_32x4d'
    num_classes = 4

    if torch.cuda.is_available():
        device = 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = 'mps'
    else:
        device = 'cpu'

TARGET_COLS = ['healthy', 'multiple_diseases', 'rust', 'scab']
CLASS_COLORS = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(CFG.seed)

NUM_WORKERS = 4 if CFG.device == 'cuda' else 0
PIN_MEMORY = (CFG.device == 'cuda')

print(f"Устройство: {CFG.device}")
print(f"Модель:     {CFG.model_name}")
print(f"Размер:     {CFG.img_h}x{CFG.img_w}")
print(f"Эпох:       {CFG.epochs} (S1) + {CFG.epochs_distill} (S2)")


## Загрузка данных

In [ ]:
train_df = pd.read_csv(os.path.join(CFG.data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(CFG.data_dir, 'test.csv'))

print(f"Train: {len(train_df)} изображений")
print(f"Test:  {len(test_df)} изображений")
train_df.head()


## EDA

In [ ]:
# --- Распределение классов ---
class_counts = train_df[TARGET_COLS].sum().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(class_counts.index, class_counts.values, color=CLASS_COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Количество изображений по классам', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Классы')
axes[0].set_ylabel('Количество')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+8, str(val), ha='center', fontsize=13, fontweight='bold')

axes[1].pie(class_counts.values, labels=class_counts.index, colors=CLASS_COLORS,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Доля каждого класса', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# --- Примеры изображений ---
fig, axes = plt.subplots(4, 5, figsize=(25, 18))
for row, (cls, color) in enumerate(zip(TARGET_COLS, CLASS_COLORS)):
    samples = train_df[train_df[cls] == 1].sample(5, random_state=CFG.seed)
    for col_idx, (_, sample) in enumerate(samples.iterrows()):
        img_path = os.path.join(CFG.data_dir, 'images', f"{sample['image_id']}.jpg")
        img = Image.open(img_path)
        axes[row][col_idx].imshow(img)
        axes[row][col_idx].set_title(f"{cls}: {sample['image_id']}", fontsize=9)
        axes[row][col_idx].axis('off')
plt.suptitle('Примеры изображений', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Поиск дубликатов (зашумлённые метки)

Одинаковые изображения с разными метками — ключевая проблема. dHash для поиска, группировка для K-Fold.

In [ ]:
def compute_dhash(img_path, hash_size=8):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (hash_size + 1, hash_size))
    diff = img[:, 1:] > img[:, :-1]
    return diff.tobytes()

hash_dict = defaultdict(list)
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Хеширование'):
    h = compute_dhash(os.path.join(CFG.data_dir, 'images', f"{row['image_id']}.jpg"))
    hash_dict[h].append(row['image_id'])

duplicates = {h: ids for h, ids in hash_dict.items() if len(ids) > 1}
print(f"Групп дубликатов: {len(duplicates)}")

for group_idx, (h, ids) in enumerate(duplicates.items()):
    print(f"\nГруппа {group_idx+1}:")
    for img_id in ids:
        row = train_df[train_df['image_id'] == img_id].iloc[0]
        print(f"  {img_id} -> {TARGET_COLS[row[TARGET_COLS].values.argmax()]}")

train_df['dup_group'] = train_df['image_id'].astype(str)
for group_idx, ids in enumerate(duplicates.values()):
    train_df.loc[train_df['image_id'].isin(ids), 'dup_group'] = f"dup_{group_idx}"
print(f"\nУникальных групп: {train_df['dup_group'].nunique()}")


In [ ]:
if duplicates:
    for group_idx, ids in enumerate(list(duplicates.values())[:3]):
        fig, axes = plt.subplots(1, len(ids), figsize=(5*len(ids), 5))
        if not isinstance(axes, np.ndarray): axes = [axes]
        for ax, img_id in zip(axes, ids):
            img = Image.open(os.path.join(CFG.data_dir, 'images', f"{img_id}.jpg"))
            ax.imshow(img)
            row = train_df[train_df['image_id'] == img_id].iloc[0]
            ax.set_title(f"{img_id}\n{TARGET_COLS[row[TARGET_COLS].values.argmax()]}", fontsize=11, fontweight='bold')
            ax.axis('off')
        plt.suptitle(f'Дубликаты группа {group_idx+1}', fontsize=13)
        plt.tight_layout()
        plt.show()


## K-Fold разбиение

In [ ]:
train_df['label'] = train_df[TARGET_COLS].values.argmax(axis=1)

sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(sgkf.split(train_df, train_df['label'], train_df['dup_group'])):
    train_df.loc[val_idx, 'fold'] = fold_idx

leak = (train_df.groupby('dup_group')['fold'].nunique() > 1).sum()
print(f"Утечек дубликатов: {leak}")

for fold in range(CFG.n_folds):
    fold_df = train_df[train_df['fold'] == fold]
    counts = fold_df['label'].value_counts().sort_index()
    parts = [f"{TARGET_COLS[i]}={counts.get(i,0)}" for i in range(4)]
    print(f"  Fold {fold}: {len(fold_df):4d} | {' | '.join(parts)}")


## Аугментации

In [ ]:
def get_train_transforms(img_h, img_w):
    return A.Compose([
        A.Resize(height=img_h, width=img_w),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=35,
                           border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_valid_transforms(img_h, img_w):
    return A.Compose([
        A.Resize(height=img_h, width=img_w),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

print(f"Train: {len(get_train_transforms(CFG.img_h, CFG.img_w))} аугментаций")


In [ ]:
class PlantDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, is_test=False, soft_labels=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
        self.is_test = is_test
        self.soft_labels = soft_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(os.path.join(self.img_dir, f"{row['image_id']}.jpg"))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transforms:
            img = self.transforms(image=img)['image']
        if self.is_test:
            return img
        if self.soft_labels is not None:
            return img, torch.tensor(self.soft_labels[idx], dtype=torch.float32)
        return img, torch.tensor(int(row['label']), dtype=torch.long)


## Модель

**SE-ResNeXt50** — SE-блок перевзвешивает каналы по важности, ResNeXt использует групповые свёртки.

In [ ]:
class PlantModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

model_test = PlantModel(CFG.model_name, CFG.num_classes, pretrained=False)
dummy = torch.randn(1, 3, CFG.img_h, CFG.img_w)
print(f"Модель: {CFG.model_name}")
print(f"Вход:   {dummy.shape} → Выход: {model_test(dummy).shape}")
print(f"Параметров: {sum(p.numel() for p in model_test.parameters()):,}")
del model_test, dummy


## Функции обучения

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scheduler=None, use_soft_labels=False):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc='  Train', leave=False)
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        if use_soft_labels:
            log_probs = torch.nn.functional.log_softmax(outputs, dim=1)
            loss = torch.nn.functional.kl_div(log_probs, labels, reduction='batchmean')
        else:
            loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{running_loss/(batch_idx+1):.4f}'})

    return running_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc='  Valid', leave=False):
        images = images.to(device)
        probs = torch.softmax(model(images), dim=1)
        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    labels_onehot = np.eye(CFG.num_classes)[all_labels]
    auc_score = roc_auc_score(labels_onehot, all_preds, multi_class='ovr', average='macro')
    return auc_score, all_preds

print('Функции определены.')


## Stage 1: Обучение 5-Fold с CrossEntropy

- **CrossEntropyLoss** — простой и эффективный
- **OneCycleLR** — warmup + cosine decay
- Сохраняем OOF-предсказания для дистилляции


In [ ]:
oof_preds = np.zeros((len(train_df), CFG.num_classes))
fold_scores = []
histories = []
training_log = []

total_start = time.time()

for fold in range(CFG.n_folds):
    print(f"\n{'='*65}")
    print(f'  FOLD {fold+1}/{CFG.n_folds} — Stage 1 (CrossEntropy)')
    print(f"{'='*65}")
    fold_start = time.time()

    train_fold = train_df[train_df['fold'] != fold].copy()
    valid_fold = train_df[train_df['fold'] == fold].copy()
    print(f'  Train: {len(train_fold)}, Valid: {len(valid_fold)}')

    img_dir = os.path.join(CFG.data_dir, 'images')
    train_dataset = PlantDataset(train_fold, img_dir, get_train_transforms(CFG.img_h, CFG.img_w))
    valid_dataset = PlantDataset(valid_fold, img_dir, get_valid_transforms(CFG.img_h, CFG.img_w))

    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=CFG.batch_size*2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = PlantModel(CFG.model_name, CFG.num_classes, pretrained=True).to(CFG.device)
    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CFG.lr, epochs=CFG.epochs,
        steps_per_epoch=len(train_loader), pct_start=0.1)
    criterion = nn.CrossEntropyLoss()

    best_auc, best_epoch, best_preds = 0, 0, None
    history = {'train_loss': [], 'valid_auc': []}

    for epoch in range(CFG.epochs):
        lr = optimizer.param_groups[0]['lr']
        t = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, CFG.device, scheduler)
        valid_auc, valid_preds = validate(model, valid_loader, CFG.device)

        history['train_loss'].append(train_loss)
        history['valid_auc'].append(valid_auc)
        training_log.append({'stage': 1, 'fold': fold, 'epoch': epoch+1,
                             'train_loss': train_loss, 'valid_auc': valid_auc, 'lr': lr})

        improved = valid_auc > best_auc
        marker = ' <<< BEST' if improved else ''
        print(f'  Epoch {epoch+1:2d}/{CFG.epochs} | Loss: {train_loss:.4f} | AUC: {valid_auc:.5f} | '
              f'lr: {lr:.2e} | {time.time()-t:.0f}s{marker}')

        if improved:
            best_auc = valid_auc
            best_epoch = epoch + 1
            best_preds = valid_preds.copy()
            torch.save(model.state_dict(), f'model_fold{fold}.pth')

    oof_preds[train_df[train_df['fold']==fold].index] = best_preds
    fold_scores.append(best_auc)
    histories.append(history)
    print(f'  Fold {fold+1}: AUC={best_auc:.5f} (epoch {best_epoch}), {(time.time()-fold_start)/60:.1f} мин')

    del model, optimizer, scheduler, train_loader, valid_loader
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\n{'='*65}")
print(f'  STAGE 1: Среднее AUC = {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}')
print(f'  Время: {(time.time()-total_start)/60:.1f} мин')
print(f"{'='*65}")


In [ ]:
log_df = pd.DataFrame(training_log)
os.makedirs('logs', exist_ok=True)
log_df.to_csv('logs/training_log.csv', index=False)
print(f"Лог сохранён: logs/training_log.csv ({len(log_df)} строк)")
for i, s in enumerate(fold_scores):
    print(f"  Fold {i+1}: AUC = {s:.5f}")
print(f"  Среднее: {np.mean(fold_scores):.5f}")


## Stage 2: Knowledge Distillation

Мягкие метки `soft = 0.3 * OOF + 0.7 * original` — модель перестаёт зубрить неправильные метки.

In [ ]:
original_onehot = np.eye(CFG.num_classes)[train_df['label'].values]
soft_labels = CFG.soft_label_ratio * oof_preds + (1 - CFG.soft_label_ratio) * original_onehot

print('Мягкие метки (примеры):')
for i in range(3):
    orig = TARGET_COLS[train_df['label'].values[i]]
    print(f'  {train_df["image_id"].values[i]:12s} | {orig:12s} | soft: {soft_labels[i].round(3)}')

distill_scores = []

for fold in range(CFG.n_folds):
    print(f"\n{'='*65}")
    print(f'  FOLD {fold+1}/{CFG.n_folds} — Stage 2 (Distillation)')
    print(f"{'='*65}")

    train_fold = train_df[train_df['fold'] != fold].copy()
    valid_fold = train_df[train_df['fold'] == fold].copy()
    train_soft = soft_labels[train_fold.index.values]

    img_dir = os.path.join(CFG.data_dir, 'images')
    train_dataset = PlantDataset(train_fold, img_dir, get_train_transforms(CFG.img_h, CFG.img_w), soft_labels=train_soft)
    valid_dataset = PlantDataset(valid_fold, img_dir, get_valid_transforms(CFG.img_h, CFG.img_w))

    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=CFG.batch_size*2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = PlantModel(CFG.model_name, CFG.num_classes, pretrained=False)
    model.load_state_dict(torch.load(f'model_fold{fold}.pth', map_location=CFG.device))
    model.to(CFG.device)

    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr * 0.3, weight_decay=CFG.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs_distill, eta_min=1e-6)

    best_auc, best_preds = 0, None

    for epoch in range(CFG.epochs_distill):
        t = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, None, CFG.device, use_soft_labels=True)
        scheduler.step()
        valid_auc, valid_preds = validate(model, valid_loader, CFG.device)

        training_log.append({'stage': 2, 'fold': fold, 'epoch': epoch+1,
                             'train_loss': train_loss, 'valid_auc': valid_auc,
                             'lr': optimizer.param_groups[0]['lr']})

        improved = valid_auc > best_auc
        marker = ' <<< BEST' if improved else ''
        print(f'  Epoch {epoch+1:2d}/{CFG.epochs_distill} | Loss: {train_loss:.4f} | AUC: {valid_auc:.5f} | {time.time()-t:.0f}s{marker}')

        if improved:
            best_auc = valid_auc
            best_preds = valid_preds.copy()
            torch.save(model.state_dict(), f'model_fold{fold}_distill.pth')

    distill_scores.append(best_auc)
    print(f'  Fold {fold+1}: AUC={best_auc:.5f}')

    del model, optimizer, scheduler, train_loader, valid_loader
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\n{'='*65}")
for i in range(CFG.n_folds):
    print(f'  Fold {i+1}: S1={fold_scores[i]:.5f} → S2={distill_scores[i]:.5f}')
print(f"  Stage 1: {np.mean(fold_scores):.5f} → Stage 2: {np.mean(distill_scores):.5f}")
print(f"{'='*65}")

log_df = pd.DataFrame(training_log)
log_df.to_csv('logs/training_log.csv', index=False)
print(f'Лог обновлён: {len(log_df)} строк')


## Анализ результатов

In [ ]:
# Кривые обучения
fig, axes = plt.subplots(CFG.n_folds, 2, figsize=(14, 3*CFG.n_folds))
for fold in range(CFG.n_folds):
    h = histories[fold]
    epochs_range = range(1, len(h['train_loss'])+1)
    axes[fold][0].plot(epochs_range, h['train_loss'], 'b-', lw=2, label='Train Loss')
    axes[fold][0].set_title(f'Fold {fold+1} — Loss')
    axes[fold][0].legend()
    axes[fold][0].grid(alpha=0.3)
    axes[fold][1].plot(epochs_range, h['valid_auc'], 'g-', lw=2, marker='o', ms=3)
    axes[fold][1].set_title(f'Fold {fold+1} — AUC (best: {fold_scores[fold]:.5f})')
    axes[fold][1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Confusion Matrix
oof_labels = train_df['label'].values
oof_pred_labels = oof_preds.argmax(axis=1)
cm = confusion_matrix(oof_labels, oof_pred_labels)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_COLS, yticklabels=TARGET_COLS, ax=ax)
ax.set_xlabel('Предсказано')
ax.set_ylabel('Истинное')
ax.set_title(f'Confusion Matrix (OOF AUC: {np.mean(fold_scores):.5f})')
plt.tight_layout()
plt.show()


## Инференс: 4 TTA + ансамбль 5 фолдов

4 детерминированных TTA (оригинал + 3 отражения). Веса фолдов по AUC.

In [ ]:
def get_tta_transforms(img_h, img_w):
    base = [A.Resize(height=img_h, width=img_w)]
    norm = [A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()]
    return [
        A.Compose(base + norm),                                                      # оригинал
        A.Compose(base + [A.HorizontalFlip(p=1.0)] + norm),                         # отражение H
        A.Compose(base + [A.VerticalFlip(p=1.0)] + norm),                           # отражение V
        A.Compose(base + [A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)] + norm),  # оба
    ]

@torch.no_grad()
def predict_with_tta(model, df, img_dir, device, img_h, img_w):
    model.eval()
    all_tta = []
    for tta_idx, transform in enumerate(get_tta_transforms(img_h, img_w)):
        dataset = PlantDataset(df, img_dir, transform, is_test=True)
        loader = DataLoader(dataset, batch_size=CFG.batch_size*2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        preds = []
        for images in tqdm(loader, desc=f'  TTA {tta_idx+1}/4', leave=False):
            probs = torch.softmax(model(images.to(device)), dim=1)
            preds.append(probs.cpu().numpy())
        all_tta.append(np.concatenate(preds))
    return np.mean(all_tta, axis=0)

# Выбираем модели
if os.path.exists('model_fold0_distill.pth'):
    model_suffix = '_distill'
    scores_for_weights = distill_scores
    print('Модели: Stage 2 (Distillation)')
else:
    model_suffix = ''
    scores_for_weights = fold_scores
    print('Модели: Stage 1')

# Веса фолдов по AUC
fold_weights = np.array(scores_for_weights, dtype=np.float64)
fold_weights = fold_weights / fold_weights.sum()
print('Веса фолдов:', {f'Fold {i+1}': f'{w:.4f}' for i, w in enumerate(fold_weights)})

test_preds = np.zeros((len(test_df), CFG.num_classes))
img_dir = os.path.join(CFG.data_dir, 'images')

for fold in range(CFG.n_folds):
    print(f'\nFold {fold+1}/{CFG.n_folds}')
    model = PlantModel(CFG.model_name, CFG.num_classes, pretrained=False)
    model.load_state_dict(torch.load(f'model_fold{fold}{model_suffix}.pth', map_location=CFG.device))
    model.to(CFG.device)
    test_preds += predict_with_tta(model, test_df, img_dir, CFG.device, CFG.img_h, CFG.img_w) * fold_weights[fold]
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\nСумма вероятностей: {test_preds[0].sum():.4f}')


## Submission

In [ ]:
submission = pd.DataFrame({
    'image_id': test_df['image_id'],
    'healthy': test_preds[:, 0],
    'multiple_diseases': test_preds[:, 1],
    'rust': test_preds[:, 2],
    'scab': test_preds[:, 3],
})
submission.to_csv('submission.csv', index=False)
print(f"submission.csv сохранён ({len(submission)} строк)")
submission.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pred_classes = test_preds.argmax(axis=1)
class_names = [TARGET_COLS[c] for c in pred_classes]
counts = pd.Series(class_names).value_counts()
bars = axes[0].bar(counts.index, counts.values,
                   color=[CLASS_COLORS[TARGET_COLS.index(c)] for c in counts.index])
axes[0].set_title('Распределение предсказаний', fontsize=13, fontweight='bold')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+3, str(val), ha='center', fontweight='bold')

axes[1].hist(test_preds.max(axis=1), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Уверенность модели', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Max probability')
plt.tight_layout()
plt.show()

conf = test_preds.max(axis=1)
print(f"Средняя уверенность: {conf.mean():.4f}")
print(f"< 0.90: {(conf<0.90).sum()} изображений")


---
## Итоги

| Компонент | Выбор |
|---|---|
| Модель | SE-ResNeXt50 |
| Размер | 320×512 |
| Loss | CrossEntropyLoss |
| Stage 1 | 20 эпох, OneCycleLR |
| Stage 2 | Knowledge Distillation (soft labels 0.3/0.7) |
| TTA | 4 варианта (flips) |
| Ансамбль | 5 фолдов, веса по AUC |
